### Comparaison des Modélisations pour la Fréquence et les Coûts des Sinistres

Dans cette section, nous allons comparer différentes approches de modélisation afin d'évaluer deux aspects clés des sinistres :

1. **La fréquence des sinistres** : Modélisation du nombre de sinistres survenus pour chaque contrat.
2. **Les coûts des sinistres** : Estimation des montants associés aux sinistres.

L'objectif est d'identifier les modèles les plus performants pour chaque aspect, en utilisant des métriques d'évaluation adaptées.

In [6]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import numpy as np
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [7]:
freq = pd.read_parquet("data/raw/freMTPLfreq.parquet")
sev = pd.read_parquet("data/raw/freMTPLsev.parquet")

In [ ]:
#Combiner les 2 bases de données

freq['PolicyID'] = freq['PolicyID'].astype(str)
sev['PolicyID'] = sev['PolicyID'].astype(str)

# Agrégation de sev : un contrat peut avoir plusieurs sinistres
sev_agg = (
    sev.groupby("PolicyID")
       .agg(
           ClaimAmount=("ClaimAmount", "sum"),   # coût total des sinistres du contrat
           ClaimNb_sev=("ClaimAmount", "size")   # nombre de sinistres déclarés dans sev
       )
       .reset_index()
)

print(len(sev), "lignes sev brutes")
print(len(sev_agg), "lignes sev agrégées (1 par PolicyID)")

merged = pd.merge(freq, sev_agg, on="PolicyID", how="left")

# Remplacer les NaN (contrats sans sinistre) par 0
merged["ClaimAmount"] = merged["ClaimAmount"].fillna(0)
merged["ClaimNb_sev"] = merged["ClaimNb_sev"].fillna(0)

print("Nb de lignes freq :", len(freq))
print("Nb de lignes merged :", len(merged))


16181 lignes sev brutes
15390 lignes sev agrégées (1 par PolicyID)
Nb de lignes freq : 413169
Nb de lignes merged : 413169


Index(['PolicyID', 'ClaimNb', 'Exposure', 'Power', 'CarAge', 'DriverAge',
       'Brand', 'Gas', 'Region', 'Density', 'ClaimAmount', 'ClaimNb_sev'],
      dtype='object')

# Création de différentes classes actuarielles nécessaires pour notre modélisation


In [15]:
# 1) Contrôles de base
print(merged[["PolicyID","ClaimNb","Exposure","ClaimAmount"]].head())
print(merged[["ClaimNb","Exposure","ClaimAmount"]].describe())

# 2) Variables actuarielles de base
merged["Freq"] = merged["ClaimNb"] / merged["Exposure"]          # fréquence annuelle
merged["PurePremium"] = merged["ClaimAmount"] / merged["Exposure"]  # prime pure
merged["AvgClaim"] = np.where(                                    # coût moyen conditionnel
    merged["ClaimNb"] > 0,
    merged["ClaimAmount"] / merged["ClaimNb"],
    np.nan
)

# 3) Quelques ratios globaux (à commenter dans le rapport)
tot_expo = merged["Exposure"].sum()
tot_claims = merged["ClaimNb"].sum()
tot_amount = merged["ClaimAmount"].sum()

freq_globale = tot_claims / tot_expo
pure_prem_globale = tot_amount / tot_expo
avgclaim_globale = tot_amount / tot_claims

print("Fréquence globale :", freq_globale)
print("Prime pure globale :", pure_prem_globale)
print("Coût moyen global :", avgclaim_globale)
print(f"% contrats sans sinistre: {(merged['ClaimNb']==0).mean()*100:.1f}%")

merged.columns

  PolicyID  ClaimNb  Exposure  ClaimAmount
0        1        0      0.09          0.0
1        2        0      0.84          0.0
2        3        0      0.52          0.0
3        4        0      0.45          0.0
4        5        0      0.15          0.0
             ClaimNb       Exposure   ClaimAmount
count  413169.000000  413169.000000  4.131690e+05
mean        0.039163       0.561088  8.341642e+01
std         0.204053       0.369477  4.192526e+03
min         0.000000       0.002732  0.000000e+00
25%         0.000000       0.200000  0.000000e+00
50%         0.000000       0.540000  0.000000e+00
75%         0.000000       1.000000  0.000000e+00
max         4.000000       1.990000  2.036833e+06
Fréquence globale : 0.06979858984933181
Prime pure globale : 148.66904231188673
Coût moyen global : 2129.9720042024596
% contrats sans sinistre: 96.3%


Index(['PolicyID', 'ClaimNb', 'Exposure', 'Power', 'CarAge', 'DriverAge',
       'Brand', 'Gas', 'Region', 'Density', 'ClaimAmount', 'ClaimNb_sev',
       'Freq', 'PurePremium', 'AvgClaim'],
      dtype='object')

# Premières approches de modélisation avec loi Poisson pour la fréquence et loi Gamma pour la sévérité

In [18]:

# ============================================================
# 1. Préparation des données et création des classes actuarielles
# ============================================================

# On garde uniquement les contrats exposés
df_glm = merged[merged["Exposure"] > 0].copy()

# Variables de classes si besoin
bins_driver = [17, 25, 30, 40, 50, 60, 120]
labels_driver = ["<25", "25-29", "30-39", "40-49", "50-59", "60+"]
df_glm["DriverAgeClass"] = pd.cut(df_glm["DriverAge"], bins=bins_driver, labels=labels_driver)

bins_car = [-1, 1, 5, 10, 20, 200]
labels_car = ["0", "1-4", "5-9", "10-19", "20+"]
df_glm["CarAgeClass"] = pd.cut(df_glm["CarAge"], bins=bins_car, labels=labels_car)

bins_dens = [0, 50, 200, 500, 2000, 10000]
labels_dens = ["rural", "peri-urbain", "petite ville", "ville", "urbain dense"]
df_glm["DensityClass"] = pd.cut(df_glm["Density"], bins=bins_dens, labels=labels_dens)

# Déclaration des qualitatives
for col in ["Power", "Brand", "Gas", "Region",
            "DriverAgeClass", "CarAgeClass", "DensityClass"]:
    df_glm[col] = df_glm[col].astype("category")

print("Taille base GLM :", len(df_glm))

# ============================================================
# 2. GLM Poisson pour la fréquence de sinistres
# ============================================================

formula_freq = (
    "ClaimNb ~ C(DriverAgeClass) + C(CarAgeClass) "
    "+ C(Power) + C(Region) + C(Gas) + C(DensityClass)"
)

model_freq = smf.glm(
    formula=formula_freq,
    data=df_glm,
    family=sm.families.Poisson(),
    offset=np.log(df_glm["Exposure"])
)
result_freq = model_freq.fit()
print(result_freq.summary())

# Fréquence prédite par contrat
df_glm["lambda_hat"] = result_freq.predict(df_glm, offset=np.log(df_glm["Exposure"]))

# Contrôle global
freq_obs = df_glm["ClaimNb"].sum() / df_glm["Exposure"].sum()
freq_hat = df_glm["lambda_hat"].sum() / df_glm["Exposure"].sum()
print(f"Fréquence observée : {freq_obs:.5f}")
print(f"Fréquence prédite  : {freq_hat:.5f}")

# ============================================================
# 3. GLM Gamma pour la sévérité (coût moyen conditionnel)
# ============================================================

# Base des contrats sinistrés
df_sev = df_glm[df_glm["ClaimNb"] > 0].copy()
df_sev["AvgClaim"] = df_sev["ClaimAmount"] / df_sev["ClaimNb"]

formula_sev = (
    "AvgClaim ~ C(DriverAgeClass) + C(CarAgeClass) "
    "+ C(Power) + C(Region) + C(Gas) + C(DensityClass)"
)

model_sev = smf.glm(
    formula=formula_sev,
    data=df_sev,
    family=sm.families.Gamma(sm.families.links.log())
)
result_sev = model_sev.fit()
print(result_sev.summary())

df_sev["cost_hat"] = result_sev.predict(df_sev)

# ============================================================
# 4. Prime pure modélisée = fréquence prédite × coût moyen prédit
# ============================================================

# Rattacher cost_hat au niveau contrat
df_glm = df_glm.merge(
    df_sev[["PolicyID", "cost_hat"]],
    on="PolicyID",
    how="left"
)

df_glm["PurePremium_hat"] = df_glm["lambda_hat"] * df_glm["cost_hat"]

# ============================================================
# 5. Comparaison observé / modélisé par segment (ex : âge conducteur)
# ============================================================

comp_age = (
    df_glm
    .groupby("DriverAgeClass")
    .agg(
        expo=("Exposure", "sum"),
        pure_obs=("PurePremium", "mean"),
        pure_hat=("PurePremium_hat", "mean"),
        freq_obs=("Freq", "mean")
    )
    .reset_index()
)

print("Comparaison prime pure observée / modélisée par classe d'âge conducteur :")
print(comp_age)



Taille base GLM : 413169
                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:               395215
Model:                            GLM   Df Residuals:                   395180
Model Family:                 Poisson   Df Model:                           34
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -64446.
Date:                Wed, 03 Dec 2025   Deviance:                       99098.
Time:                        12:30:25   Pearson chi2:                 6.89e+05
No. Iterations:                     7   Pseudo R-squ. (CS):           0.004092
Covariance Type:            nonrobust                                         
                                      coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------

/opt/python/lib/python3.13/site-packages/statsmodels/genmod/families/links.py:13: FutureWarning: The log link alias is deprecated. Use Log instead. The log link alias will be removed after the 0.15.0 release.
  warnings.warn(


                 Generalized Linear Model Regression Results                  
Dep. Variable:               AvgClaim   No. Observations:                14677
Model:                            GLM   Df Residuals:                    14642
Model Family:                   Gamma   Df Model:                           34
Link Function:                    log   Scale:                          21.755
Method:                          IRLS   Log-Likelihood:            -1.4846e+05
Date:                Wed, 03 Dec 2025   Deviance:                       22307.
Time:                        12:30:36   Pearson chi2:                 3.19e+05
No. Iterations:                    42   Pseudo R-squ. (CS):           0.005517
Covariance Type:            nonrobust                                         
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept 

/tmp/ipykernel_45336/3752373817.py:97: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("DriverAgeClass")


- **Modèle de fréquence (Poisson)**  
  Le nombre de sinistres par contrat est modélisé par un GLM de famille Poisson avec lien logarithmique et offset `log(Exposure)`. Le modèle reproduit correctement la fréquence moyenne du portefeuille, mais la pseudo-\(R^2\) est faible et la déviance résiduelle importante, ce qui met en évidence une sur‑dispersion et un pouvoir explicatif limité. Les effets estimés restent cohérents avec l’intuition actuarielle : la fréquence décroît avec l’âge du conducteur et augmente dans les zones urbaines denses.

- **Modèle de sévérité (Gamma)**  
  Le coût moyen conditionnel par sinistre (`AvgClaim`) est modélisé par un GLM Gamma à lien log sur la sous‑population sinistrée. Là encore, la pseudo-\(R^2\) est très faible et le chi‑deux de Pearson signale une forte variabilité résiduelle, typique des montants de sinistres. Les coefficients montrent toutefois une tendance à des coûts moyens plus faibles pour les conducteurs plus âgés, tandis que les autres facteurs (âge véhicule, puissance, région) ont des effets plus modérés.

## Comparaison entre GLM Poisson et GLM Négative Binomiale (pour le fréquence)

In [33]:

df = merged.copy()

# Création des classes actuarielles (si pas déjà fait)
bins_driver = [17, 25, 30, 40, 50, 60, 120]
labels_driver = ["<25", "25-29", "30-39", "40-49", "50-59", "60+"]
df["DriverAgeClass"] = pd.cut(df["DriverAge"], bins=bins_driver, labels=labels_driver)

bins_car = [-1, 1, 5, 10, 20, 200]
labels_car = ["0", "1-4", "5-9", "10-19", "20+"]
df["CarAgeClass"] = pd.cut(df["CarAge"], bins=bins_car, labels=labels_car)

bins_dens = [0, 50, 200, 500, 2000, 10000]
labels_dens = ["rural", "peri-urbain", "petite ville", "ville", "urbain dense"]
df["DensityClass"] = pd.cut(df["Density"], bins=bins_dens, labels=labels_dens)

# On ne garde que les contrats exposés
df = df[df["Exposure"] > 0].copy()

for col in ["Power", "Brand", "Gas", "Region",
            "DriverAgeClass", "CarAgeClass", "DensityClass"]:
    df[col] = df[col].astype("category")

# =========================
# 2. Split 75 % / 25 %
# =========================

df_train, df_test = train_test_split(df, test_size=0.25, random_state=123)

print("Taille train :", len(df_train))
print("Taille test  :", len(df_test))

# =========================
# 3. Modèle Poisson complet
# =========================

formula_full = (
    "ClaimNb ~ C(DriverAgeClass) + C(CarAgeClass) "
    "+ C(Power) + C(Region) + C(DensityClass)"
)

model_full = smf.glm(
    formula=formula_full,
    data=df_train,
    family=sm.families.Poisson(),
    offset=np.log(df_train["Exposure"])
)
res_full = model_full.fit()
print("AIC Poisson complet (train) :", res_full.aic)


# =========================
# 3. Binomiale négative complète (train)
# =========================

nb_family = sm.families.NegativeBinomial()

model_nb = smf.glm(
    formula=formula_full,
    data=df_train,
    family=nb_family,
    offset=np.log(df_train["Exposure"])
)
res_nb = model_nb.fit()
print("AIC Binomiale négative (train) :", res_nb.aic)

# =========================
# 4. Fonction déviance Poisson sur le test
# =========================

def poisson_deviance(result, df_eval):
    mu = result.predict(df_eval, offset=np.log(df_eval["Exposure"]))
    y = df_eval["ClaimNb"].to_numpy(dtype=float)
    mu = mu.to_numpy(dtype=float)

    # Retirer les NaN éventuels (ex: classes manquantes)
    mask = ~np.isnan(mu)
    y = y[mask]
    mu = mu[mask]

    eps = 1e-10
    mu_safe = np.where(mu <= 0, eps, mu)

    mask_pos = y > 0
    term1 = np.zeros_like(y, dtype=float)
    term1[mask_pos] = y[mask_pos] * np.log((y[mask_pos] + eps) / mu_safe[mask_pos])

    dev = 2 * np.sum(term1 - (y - mu_safe))
    return dev, mask.sum()

# =========================
# 5. Déviance Poisson sur le test
# =========================

dev_pois, n_pois = poisson_deviance(res_pois, df_test)
dev_nb,   n_nb   = poisson_deviance(res_nb,  df_test)

print(f"Déviance (test) Poisson, n={n_pois} :", dev_pois)
print(f"Déviance (test) NB,      n={n_nb}   :", dev_nb)

# =========================
# 6. Modèle retenu
# =========================

if (res_nb.aic < res_pois.aic) and (dev_nb < dev_pois):
    res_best = res_nb
    best_name = "Binomiale négative"
else:
    res_best = res_pois
    best_name = "Poisson"

print("Modèle retenu pour la tarification (comparaison Poisson vs NB) :", best_name)

# Fréquence prédite par contrat avec le modèle retenu
df["lambda_hat_best"] = res_best.predict(df, offset=np.log(df["Exposure"]))


Taille train : 309876
Taille test  : 103293
AIC Poisson complet (train) : 96388.31785946968


/opt/python/lib/python3.13/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


AIC Binomiale négative (train) : 96162.97515820694
Déviance (test) Poisson, n=98804 : 25146.206162684364
Déviance (test) NB,      n=98804   : 25158.629056747195
Modèle retenu pour la tarification (comparaison Poisson vs NB) : Poisson


In [30]:


# =========================
# 1. Base sévérité (sinistrés)
# =========================

df_sev = merged[(merged["Exposure"] > 0) & (merged["ClaimNb"] > 0)].copy()
df_sev["AvgClaim"] = df_sev["ClaimAmount"] / df_sev["ClaimNb"]

# mêmes classes que pour la fréquence
bins_driver = [17, 25, 30, 40, 50, 60, 120]
labels_driver = ["<25", "25-29", "30-39", "40-49", "50-59", "60+"]
df_sev["DriverAgeClass"] = pd.cut(df_sev["DriverAge"], bins=bins_driver, labels=labels_driver)

bins_car = [-1, 1, 5, 10, 20, 200]
labels_car = ["0", "1-4", "5-9", "10-19", "20+"]
df_sev["CarAgeClass"] = pd.cut(df_sev["CarAge"], bins=bins_car, labels=labels_car)

bins_dens = [0, 50, 200, 500, 2000, 10000]
labels_dens = ["rural", "peri-urbain", "petite ville", "ville", "urbain dense"]
df_sev["DensityClass"] = pd.cut(df_sev["Density"], bins=bins_dens, labels=labels_dens)

for col in ["Power", "Brand", "Gas", "Region",
            "DriverAgeClass", "CarAgeClass", "DensityClass"]:
    df_sev[col] = df_sev[col].astype("category")

# 75 % / 25 %
sev_train, sev_test = train_test_split(df_sev, test_size=0.25, random_state=123)

print("Taille train sévérité :", len(sev_train))
print("Taille test sévérité  :", len(sev_test))

formula_sev = (
    "AvgClaim ~ C(DriverAgeClass) + C(CarAgeClass) "
    "+ C(Power) + C(Region) + C(DensityClass)"
)

# =========================
# 2. GLM Gamma (lien log)
# =========================

model_gamma = smf.glm(
    formula=formula_sev,
    data=sev_train,
    family=sm.families.Gamma(sm.families.links.log())
)
res_gamma = model_gamma.fit()
print("AIC Gamma (train) :", res_gamma.aic)

# =========================
# 3. GLM Lognormal (Gaussian sur log(AvgClaim))
# =========================

sev_train_ln = sev_train.copy()
sev_test_ln  = sev_test.copy()
sev_train_ln["logAvg"] = np.log(sev_train_ln["AvgClaim"])
sev_test_ln["logAvg"]  = np.log(sev_test_ln["AvgClaim"])

formula_logn = (
    "logAvg ~ C(DriverAgeClass) + C(CarAgeClass) "
    "+ C(Power) + C(Region) + C(DensityClass)"
)

model_logn = smf.ols(
    formula=formula_logn,
    data=sev_train_ln
)
res_logn = model_logn.fit()
print("AIC Lognormal (train) :", res_logn.aic)

# =========================
# 4. Déviance Gamma sur le test (critère commun)
# =========================

def gamma_deviance(mu, y):
    """Déviance Gamma (scale=1) pour vecteurs numpy positifs."""
    eps = 1e-10
    y = np.asarray(y, float)
    mu = np.asarray(mu, float)
    mask = (y > 0) & (mu > 0) & ~np.isnan(mu)
    y = y[mask]
    mu = mu[mask]
    return 2 * np.sum((y - mu) / mu - np.log(y / mu + eps)), mask.sum()

# Prédictions Gamma
mu_gamma_test = res_gamma.predict(sev_test)
dev_gamma, n_g = gamma_deviance(mu_gamma_test, sev_test["AvgClaim"])
print(f"Déviance Gamma (test, n={n_g}) :", dev_gamma)

# Prédictions Lognormal ramenées au niveau moyen (E[Y] ≈ exp(m + s²/2])
mu_logn_test_ln = res_logn.predict(sev_test_ln)
sigma2 = res_logn.scale  # variance résiduelle sur log
mu_logn_test = np.exp(mu_logn_test_ln + 0.5 * sigma2)

dev_logn, n_l = gamma_deviance(mu_logn_test, sev_test["AvgClaim"])
print(f"Déviance (critère Gamma) Lognormal (test, n={n_l}) :", dev_logn)

# =========================
# 5. Modèle de sévérité retenu
# =========================

if (res_logn.aic < res_gamma.aic) and (dev_logn < dev_gamma):
    sev_best = "Lognormal"
else:
    sev_best = "Gamma"

print("Modèle retenu pour la sévérité :", sev_best)


Taille train sévérité : 11542
Taille test sévérité  : 3848


/opt/python/lib/python3.13/site-packages/statsmodels/genmod/families/links.py:13: FutureWarning: The log link alias is deprecated. Use Log instead. The log link alias will be removed after the 0.15.0 release.
  warnings.warn(


AIC Gamma (train) : 220981.1489283473
AIC Lognormal (train) : 33319.65888087542
Déviance Gamma (test, n=3681) : 6183.523444145096
Déviance (critère Gamma) Lognormal (test, n=3681) : 6075.794312288559
Modèle retenu pour la sévérité : Lognormal


## Approche avec une loi Binomiale Négative pour la fréquence 

In [19]:
 #2) Formule du modèle
formula_freq_nb = (
    "ClaimNb ~ C(DriverAgeClass) + C(CarAgeClass) "
    "+ C(Power) + C(Region) + C(Gas) + C(DensityClass)"
)

# 3) GLM binomiale négative (variance > moyenne)
nb_family = sm.families.NegativeBinomial()  # lien log par défaut

model_freq_nb = smf.glm(
    formula=formula_freq_nb,
    data=df_glm,
    family=nb_family,
    offset=np.log(df_glm["Exposure"])
)
result_freq_nb = model_freq_nb.fit()
print(result_freq_nb.summary())

# 4) Fréquence prédite et contrôle global
df_glm["lambda_hat_nb"] = result_freq_nb.predict(df_glm, offset=np.log(df_glm["Exposure"]))

freq_obs = df_glm["ClaimNb"].sum() / df_glm["Exposure"].sum()
freq_hat_nb = df_glm["lambda_hat_nb"].sum() / df_glm["Exposure"].sum()
print(f"Fréquence observée : {freq_obs:.5f}")
print(f"Fréquence prédite (NB) : {freq_hat_nb:.5f}")

/opt/python/lib/python3.13/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:               395215
Model:                            GLM   Df Residuals:                   395180
Model Family:        NegativeBinomial   Df Model:                           34
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -64282.
Date:                Wed, 03 Dec 2025   Deviance:                       87118.
Time:                        12:38:48   Pearson chi2:                 6.70e+05
No. Iterations:                     7   Pseudo R-squ. (CS):           0.003920
Covariance Type:            nonrobust                                         
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept 

# Tarification & validation

In [34]:

# ============================================================
# 1. Préparation : base fréquence + sévérité, split 75/25
# ============================================================

df = merged.copy()
df = df[(df["Exposure"] > 0) & (df["ClaimNb"] >= 0)].copy()
df["AvgClaim"] = np.where(df["ClaimNb"] > 0,
                          df["ClaimAmount"] / df["ClaimNb"],
                          np.nan)

# Classes actuarielles
bins_driver = [17, 25, 30, 40, 50, 60, 120]
labels_driver = ["<25", "25-29", "30-39", "40-49", "50-59", "60+"]
df["DriverAgeClass"] = pd.cut(df["DriverAge"], bins=bins_driver, labels=labels_driver)

bins_car = [-1, 1, 5, 10, 20, 200]
labels_car = ["0", "1-4", "5-9", "10-19", "20+"]
df["CarAgeClass"] = pd.cut(df["CarAge"], bins=bins_car, labels=labels_car)

bins_dens = [0, 50, 200, 500, 2000, 10000]
labels_dens = ["rural", "peri-urbain", "petite ville", "ville", "urbain dense"]
df["DensityClass"] = pd.cut(df["Density"], bins=bins_dens, labels=labels_dens)

for col in ["Power", "Brand", "Gas", "Region",
            "DriverAgeClass", "CarAgeClass", "DensityClass"]:
    df[col] = df[col].astype("category")

# Split contrats pour la fréquence
df_train, df_test = train_test_split(df, test_size=0.25, random_state=123)

# Base sévérité : uniquement sinistrés du train / test
sev_train = df_train[df_train["ClaimNb"] > 0].copy()
sev_test  = df_test[df_test["ClaimNb"] > 0].copy()

# ============================================================
# 2. Modèle de fréquence : Poisson complet
# ============================================================

formula_freq = (
    "ClaimNb ~ C(DriverAgeClass) + C(CarAgeClass) "
    "+ C(Power) + C(Region) + C(Gas) + C(DensityClass)"
)

mod_freq = smf.glm(
    formula=formula_freq,
    data=df_train,
    family=sm.families.Poisson(),
    offset=np.log(df_train["Exposure"])
)
res_freq = mod_freq.fit()
print("AIC Poisson fréquence (train) :", res_freq.aic)

# Fréquence prédite sur train / test
df_train["lambda_hat"] = res_freq.predict(df_train, offset=np.log(df_train["Exposure"]))
df_test["lambda_hat"]  = res_freq.predict(df_test,  offset=np.log(df_test["Exposure"]))

# ============================================================
# 3. Modèle de sévérité : lognormal (Gaussian sur log AvgClaim)
# ============================================================

sev_train = sev_train.copy()
sev_test  = sev_test.copy()
sev_train["logAvg"] = np.log(sev_train["AvgClaim"])
sev_test["logAvg"]  = np.log(sev_test["AvgClaim"])

formula_sev = (
    "logAvg ~ C(DriverAgeClass) + C(CarAgeClass) "
    "+ C(Power) + C(Region) + C(Gas) + C(DensityClass)"
)

mod_sev = smf.ols(
    formula=formula_sev,
    data=sev_train
)
res_sev = mod_sev.fit()
print("AIC Lognormal sévérité (train) :", res_sev.aic)

# Sévérité prédite (espérance lognormale) sur train / test
sigma2 = res_sev.scale

sev_train["logAvg_hat"] = res_sev.predict(sev_train)
sev_test["logAvg_hat"]  = res_sev.predict(sev_test)

sev_train["sev_hat"] = np.exp(sev_train["logAvg_hat"] + 0.5 * sigma2)
sev_test["sev_hat"]  = np.exp(sev_test["logAvg_hat"]  + 0.5 * sigma2)

# Rattacher sev_hat aux dataframes fréquence
df_train = df_train.merge(
    sev_train[["PolicyID", "sev_hat"]],
    on="PolicyID",
    how="left"
)
df_test = df_test.merge(
    sev_test[["PolicyID", "sev_hat"]],
    on="PolicyID",
    how="left"
)

# Pour les contrats sans sinistre (pas de prédiction directe), on peut
# utiliser la sévérité moyenne globale estimée sur sev_train
sev_global = sev_train["sev_hat"].mean()
df_train["sev_hat"] = df_train["sev_hat"].fillna(sev_global)
df_test["sev_hat"]  = df_test["sev_hat"].fillna(sev_global)

# ============================================================
# 4. Prime pure modélisée et tarif
# ============================================================

# Prime pure modélisée par contrat (unité d'exposition)
df_train["PurePremium_hat"] = df_train["lambda_hat"] * df_train["sev_hat"]
df_test["PurePremium_hat"]  = df_test["lambda_hat"]  * df_test["sev_hat"]

# Chargement global (ex : +20 %)
loading = 1.20
df_train["Tarif"] = df_train["PurePremium_hat"] * loading
df_test["Tarif"]  = df_test["PurePremium_hat"]  * loading

# ============================================================
# 5. Vérification du tarif sur l’échantillon de validation
#    -> comparaison prime pure observée vs modélisée par segment
# ============================================================

group_cols = ["DriverAgeClass", "Power", "Region"]

check_test = (
    df_test
    .groupby(group_cols)
    .agg(
        Exposure_tot=("Exposure", "sum"),
        Pure_obs=("PurePremium", "mean"),
        Pure_hat=("PurePremium_hat", "mean"),
        Tarif_moy=("Tarif", "mean")
    )
    .reset_index()
)

print("Vérification sur l'échantillon de validation (quelques segments) :")
display(check_test.sort_values("Exposure_tot", ascending=False).head(15))


AIC Poisson fréquence (train) : 96302.78139129127
AIC Lognormal sévérité (train) : 33283.1920936064
Vérification sur l'échantillon de validation (quelques segments) :


/tmp/ipykernel_45336/3294415293.py:127: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(group_cols)


,DriverAgeClass,Power,Region,Exposure_tot,Pure_obs,Pure_hat,Tarif_moy
383,40-49,f,Centre,1608.721288,323.353863,74.858927,89.830713
263,30-39,f,Centre,1553.444164,305.425258,60.607201,72.728641
393,40-49,g,Centre,1473.794849,182.326372,69.724272,83.669126
623,60+,f,Centre,1446.532909,502.597644,73.471508,88.165809
513,50-59,g,Centre,1372.538418,214.761272,66.854473,80.225368
503,50-59,f,Centre,1306.773798,753.431430,68.144146,81.772975
633,60+,g,Centre,1272.075609,77.940247,68.070080,81.684095
273,30-39,g,Centre,1265.293205,148.026016,53.992147,64.790576
373,40-49,e,Centre,1123.480222,99.148069,75.652391,90.782869
253,30-39,e,Centre,1116.862079,1096.095959,62.953526,75.544232


In [35]:
# Comparaison globale sur le test
pure_obs_globale = (df_test["ClaimAmount"].sum() / df_test["Exposure"].sum())
pure_hat_globale = (df_test["PurePremium_hat"].sum() / df_test["Exposure"].sum())

print("Prime pure observée (test)  :", pure_obs_globale)
print("Prime pure modélisée (test) :", pure_hat_globale)

# Ratio modèle / observé
print("Ratio modèle / observé :", pure_hat_globale / pure_obs_globale)


Prime pure observée (test)  : 148.2704466040269
Prime pure modélisée (test) : 113.7131096281543
Ratio modèle / observé : 0.7669303777834979


In [36]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_poisson_deviance

# =========================
# 1. Préparation des données
# =========================

df = merged.copy()
df = df[df["Exposure"] > 0].copy()

# Variable réponse = fréquence
y = df["ClaimNb"] / df["Exposure"]

# Variables explicatives (toutes qualitatives ici)
X_cat = df[["DriverAgeClass", "CarAgeClass", "Power", "Region", "Gas", "DensityClass"]].astype("category")

# One-hot encoding sans drop de la catégorie de base (scikit gère la pénalisation)
enc = OneHotEncoder(drop=None, sparse_output=False)
X_encoded = enc.fit_transform(X_cat)

# Poids = exposition (offset en GLM)
sample_weight = df["Exposure"].to_numpy()

# Split 75 / 25
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X_encoded, y, sample_weight, test_size=0.25, random_state=123
)

# =========================
# 2. Régression de Poisson pénalisée (Lasso)
# =========================
# alpha = paramètre de pénalisation (à ajuster, éventuellement par grille)
alphas = [0.001, 0.01, 0.1, 1.0]
results = []

for a in alphas:
    model_lasso = PoissonRegressor(alpha=a, fit_intercept=True, max_iter=1000)
    model_lasso.fit(X_train, y_train, sample_weight=w_train)

    # Déviance de Poisson sur train et test
    mu_train = model_lasso.predict(X_train)
    mu_test  = model_lasso.predict(X_test)

    dev_train = mean_poisson_deviance(y_train, mu_train, sample_weight=w_train) * len(y_train)
    dev_test  = mean_poisson_deviance(y_test,  mu_test,  sample_weight=w_test)  * len(y_test)

    n_nonzero = np.sum(model_lasso.coef_ != 0)
    results.append((a, dev_train, dev_test, n_nonzero))
    print(f"alpha={a}  Dév.train={dev_train:.1f}  Dév.test={dev_test:.1f}  nb coeff≠0={n_nonzero}")

# Choix de l'alpha avec la plus faible déviance test
best = min(results, key=lambda t: t[2])
alpha_best = best[0]
print("alpha retenu :", alpha_best)

# =========================
# 3. Modèle final Lasso et interprétation
# =========================

model_lasso = PoissonRegressor(alpha=alpha_best, fit_intercept=True, max_iter=1000)
model_lasso.fit(X_train, y_train, sample_weight=w_train)

mu_test = model_lasso.predict(X_test)
dev_test_best = mean_poisson_deviance(y_test, mu_test, sample_weight=w_test) * len(y_test)
print("Déviance test modèle Lasso :", dev_test_best)

# Récupérer les coefficients non nuls avec leurs variables
coef = model_lasso.coef_
feature_names = enc.get_feature_names_out(X_cat.columns)

selected = [(name, c) for name, c in zip(feature_names, coef) if abs(c) > 1e-6]
selected = sorted(selected, key=lambda x: x[1], reverse=True)

print("Variables sélectionnées par le Lasso (coefs non nuls) :")
for name, c in selected:
    print(f"{name:35s}  coef={c:.3f}")


alpha=0.001  Dév.train=138351.2  Dév.test=46950.7  nb coeff≠0=41
alpha=0.01  Dév.train=138992.0  Dév.test=47106.9  nb coeff≠0=41
alpha=0.1  Dév.train=140201.7  Dév.test=47454.1  nb coeff≠0=41
alpha=1.0  Dév.train=140585.4  Dév.test=47571.6  nb coeff≠0=41
alpha retenu : 0.001
Déviance test modèle Lasso : 46950.723899183526
Variables sélectionnées par le Lasso (coefs non nuls) :
DriverAgeClass_<25                   coef=0.569
DensityClass_nan                     coef=0.216
DensityClass_urbain dense            coef=0.207
CarAgeClass_5-9                      coef=0.121
Gas_Diesel                           coef=0.108
DensityClass_ville                   coef=0.092
Power_i                              coef=0.085
Power_k                              coef=0.085
Power_j                              coef=0.076
Region_Limousin                      coef=0.071
Region_Aquitaine                     coef=0.054
Region_Ile-de-France                 coef=0.048
DriverAgeClass_25-29                 coef=0.

## Approche avec une Quasi-Poisson

In [20]:
# On part de df_glm déjà préparé (Exposure > 0, variables catégorielles, etc.)

formula_freq = (
    "ClaimNb ~ C(DriverAgeClass) + C(CarAgeClass) "
    "+ C(Power) + C(Region) + C(Gas) + C(DensityClass)"
)

# Estimation Poisson avec ajustement de dispersion (quasi-Poisson)
model_qp = smf.glm(
    formula=formula_freq,
    data=df_glm,
    family=sm.families.Poisson(),
    offset=np.log(df_glm["Exposure"])
)

result_qp = model_qp.fit(scale="X2")   # ou scale="pearson"
print(result_qp.summary())

# Les prédictions de fréquence restent les mêmes que le Poisson simple
df_glm["lambda_hat_qp"] = result_qp.predict(df_glm, offset=np.log(df_glm["Exposure"]))

                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:               395215
Model:                            GLM   Df Residuals:                   395180
Model Family:                 Poisson   Df Model:                           34
Link Function:                    Log   Scale:                          1.7431
Method:                          IRLS   Log-Likelihood:                -36972.
Date:                Wed, 03 Dec 2025   Deviance:                       99098.
Time:                        12:41:12   Pearson chi2:                 6.89e+05
No. Iterations:                     9   Pseudo R-squ. (CS):           0.002349
Covariance Type:            nonrobust                                         
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept 